# CIC IIoT 2025 Dataset Preparation
This notebook prepares the CIC IIoT 2025 dataset into a single clean CSV for modeling and XAI.

## 1. Setup

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Setup complete')

Setup complete


## 2. Paths and Dataset Discovery

In [2]:
BASE_PATH = Path('/home/ibibers/XAI_Evaluation (Exp_Pow_Action_ExpAcc) IEEE TIFS/XAI Evalation For IDS datasets (Explanatory Power,Actionability,Explanation Accuracy)/IDS_Datasets')
DATASETS_PATH = BASE_PATH / '2025 IDS Datasets'
READY_PATH = BASE_PATH / 'Ready Datasets'

DATASET_NAME = 'CIC_IIoT_2025'
DATASET_PATH = DATASETS_PATH / 'CIC IIoT dataset 2025'
OUTPUT_CSV = READY_PATH / f'{DATASET_NAME}_cleaned.csv'
PREPROCESSOR_PATH = READY_PATH / f'{DATASET_NAME}_preprocessor.joblib'
CLASS_WEIGHTS_PATH = READY_PATH / f'{DATASET_NAME}_class_weights.json'

READY_PATH.mkdir(parents=True, exist_ok=True)

csv_files = sorted([p for p in DATASET_PATH.rglob('*.csv') if p.is_file()])
print(f'Dataset path: {DATASET_PATH}')
print(f'CSV files found: {len(csv_files)}')
print('Sample files:')
for p in csv_files[:5]:
    print(f'  - {p.name}')

Dataset path: /home/ibibers/XAI_Evaluation (Exp_Pow_Action_ExpAcc) IEEE TIFS/XAI Evalation For IDS datasets (Explanatory Power,Actionability,Explanation Accuracy)/IDS_Datasets/2025 IDS Datasets/CIC IIoT dataset 2025
CSV files found: 20
Sample files:
  - attack_samples_10sec.csv
  - attack_samples_1sec.csv
  - attack_samples_2sec.csv
  - attack_samples_3sec.csv
  - attack_samples_4sec.csv


## 3. Load and Merge Raw CSV Files

In [3]:
if not csv_files:
    raise FileNotFoundError('No CSV files found for this dataset')

dataframes = []
errors = []

for csv_file in csv_files:
    try:
        df_part = pd.read_csv(csv_file, low_memory=False, on_bad_lines='skip')
        dataframes.append(df_part)
    except Exception as exc:
        errors.append((csv_file.name, str(exc)))

df_raw = pd.concat(dataframes, ignore_index=True)
print(f'Loaded files: {len(dataframes)}')
print(f'Errors: {len(errors)}')
print(f'Raw shape: {df_raw.shape}')
if errors:
    print('Sample errors:')
    for item in errors[:5]:
        print(f'  - {item[0]}: {item[1]}')

Loaded files: 20
Errors: 0
Raw shape: (685671, 94)


## 4. Basic Statistics

In [4]:
print('Columns:', len(df_raw.columns))
print('Missing values:', int(df_raw.isnull().sum().sum()))
print('Duplicate rows:', int(df_raw.duplicated().sum()))

candidate_labels = [c for c in df_raw.columns if any(k in c.lower() for k in ['label', 'class', 'attack', 'target'])]
print('Candidate label columns:', candidate_labels)

Columns: 94
Missing values: 0
Duplicate rows: 0
Candidate label columns: ['label_full', 'label1', 'label2', 'label3', 'label4']


## 5. Clean Columns and Remove Leakage Features

In [5]:
def clean_column_name(name: str) -> str:
    name = name.strip().lower()
    name = re.sub(r'[\s\-\.\/]+', '_', name)
    name = re.sub(r'[^a-z0-9_]+', '', name)
    return name

df = df_raw.copy()
df.columns = [clean_column_name(c) for c in df.columns]

leakage_patterns = [
    r'ip', r'src_ip', r'dst_ip', r'timestamp', r'time', r'date', r'uuid', r'id$',
    r'session', r'flow_id', r'flowid', r'connection_id', r'packet_id', r'index'
 ]
leakage_cols = [c for c in df.columns if any(re.search(pat, c) for pat in leakage_patterns)]
print('Leakage columns detected:', leakage_cols)
df = df.drop(columns=leakage_cols, errors='ignore')

df = df.dropna(axis=1, how='all')
print(f'Shape after leakage removal: {df.shape}')

Leakage columns detected: ['timestamp', 'timestamp_start', 'timestamp_end', 'network_ip_flags_avg', 'network_ip_flags_max', 'network_ip_flags_min', 'network_ip_flags_std_deviation', 'network_ip_length_avg', 'network_ip_length_max', 'network_ip_length_min', 'network_ip_length_std_deviation', 'network_ips_all', 'network_ips_all_count', 'network_ips_dst', 'network_ips_dst_count', 'network_ips_src', 'network_ips_src_count', 'network_time_delta_avg', 'network_time_delta_max', 'network_time_delta_min', 'network_time_delta_std_deviation']
Shape after leakage removal: (685671, 73)


## 6. Label Standardization

In [6]:
label_candidates = [c for c in df.columns if any(k in c for k in ['label', 'class', 'attack', 'target'])]
if not label_candidates:
    raise ValueError('No label column found. Please specify the label column name.')

preferred = [c for c in ['label', 'class', 'attack', 'target'] if c in df.columns]
label_col = preferred[0] if preferred else label_candidates[0]
print(f'Using label column: {label_col}')

df['label'] = df[label_col].astype(str).str.strip().str.lower()
df['label'] = df['label'].replace({'normal': 'benign'})
df = df[~df['label'].isin(['', 'nan'])].reset_index(drop=True)

extra_label_cols = [c for c in label_candidates if c != label_col]
df = df.drop(columns=extra_label_cols, errors='ignore')
if label_col != 'label':
    df = df.drop(columns=[label_col])

print(df['label'].value_counts().head(10))

Using label column: label_full
label
benign_whole-network3                                   400672
attack_recon_os-scan_whole-network                        7115
attack_recon_host-disc-tcp-ack-ping_whole-network         7056
attack_recon_vuln-scan_whole-network                      7041
attack_recon_host-disc-tcp-syn-ping_whole-network         6991
attack_recon_port-scan_whole-network                      6962
attack_recon_ping-sweep_whole-network                     6955
attack_recon_host-disc-tcp-syn-stealth_whole-network      6923
attack_recon_host-disc-arp-ping_whole-network             6915
attack_recon_host-disc-udp-ping_whole-network             6880
Name: count, dtype: int64


## 7. Remove Duplicates, Impute, and Frequency Encode

In [8]:
df = df.drop_duplicates().reset_index(drop=True)

X = df.drop(columns=['label'])
y = df['label']

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

numeric_imputer = SimpleImputer(strategy='median')
categorical_imputer = SimpleImputer(strategy='most_frequent')

if numeric_cols:
    X_num = numeric_imputer.fit_transform(X[numeric_cols])
else:
    X_num = np.empty((len(X), 0))

if categorical_cols:
    X_cat_raw = categorical_imputer.fit_transform(X[categorical_cols])
    X_cat_df = pd.DataFrame(X_cat_raw, columns=categorical_cols)
    freq_maps = {}
    for col in categorical_cols:
        freqs = X_cat_df[col].value_counts(normalize=True)
        freq_maps[col] = freqs.to_dict()
        X_cat_df[col] = X_cat_df[col].map(freq_maps[col]).fillna(0.0)
    X_cat = X_cat_df.to_numpy(dtype=float)
else:
    X_cat = np.empty((len(X), 0))
    freq_maps = {}

X_processed = np.hstack([X_num, X_cat])

feature_names = numeric_cols + categorical_cols
df_processed = pd.DataFrame(X_processed, columns=feature_names)
df_processed['label'] = y.values

print(f'Processed shape: {df_processed.shape}')

Processed shape: (384449, 69)


## 8. Class Weights (No Synthetic Data)

In [9]:
classes = np.unique(y)
if len(classes) > 1:
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
    class_weights = {cls: float(w) for cls, w in zip(classes, weights)}
else:
    class_weights = {classes[0]: 1.0}

print('Class weights:', class_weights)
with open(CLASS_WEIGHTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(class_weights, f, indent=2)

Class weights: {'attack_bruteforce_dictionary-ssh_ap': 3.5991031474096125, 'attack_bruteforce_dictionary-ssh_edge1': 2.0827297401253597, 'attack_bruteforce_dictionary-ssh_mqtt-broker': 0.4225517598400575, 'attack_bruteforce_dictionary-ssh_router': 1.0602009271439168, 'attack_bruteforce_dictionary-ssh_switch': 1.071273521683279, 'attack_bruteforce_dictionary-telnet_ap': 0.6160627009079517, 'attack_bruteforce_dictionary-telnet_edge1': 0.4710651651029803, 'attack_bruteforce_dictionary-telnet_mqtt-broker': 0.4225517598400575, 'attack_ddos_ack-frag-flood-port-1883_accelerometer-sensor': 2.1594618884457675, 'attack_ddos_ack-frag-flood-port-1883_flame-sensor': 2.486653083664823, 'attack_ddos_ack-frag-flood-port-1883_gas-sensor': 2.1824348872590202, 'attack_ddos_ack-frag-flood-port-1883_light-sensor': 2.1369674937744576, 'attack_ddos_ack-frag-flood-port-1883_motion-sensor': 2.4277973893768983, 'attack_ddos_ack-frag-flood-port-1883_mqtt-broker': 2.1149369010551333, 'attack_ddos_ack-frag-flood-p

## 9. Save Outputs
This saves the cleaned dataset and preprocessing artifacts for reproducibility.

In [10]:
df_processed.to_csv(OUTPUT_CSV, index=False)
joblib.dump({
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'freq_maps': freq_maps,
    'imputer_num': numeric_imputer,
    'imputer_cat': categorical_imputer,
    'feature_names': feature_names
}, PREPROCESSOR_PATH)

print(f'Saved cleaned CSV: {OUTPUT_CSV}')
print(f'Saved preprocessor: {PREPROCESSOR_PATH}')
print(f'Saved class weights: {CLASS_WEIGHTS_PATH}')

Saved cleaned CSV: /home/ibibers/XAI_Evaluation (Exp_Pow_Action_ExpAcc) IEEE TIFS/XAI Evalation For IDS datasets (Explanatory Power,Actionability,Explanation Accuracy)/IDS_Datasets/Ready Datasets/CIC_IIoT_2025_cleaned.csv
Saved preprocessor: /home/ibibers/XAI_Evaluation (Exp_Pow_Action_ExpAcc) IEEE TIFS/XAI Evalation For IDS datasets (Explanatory Power,Actionability,Explanation Accuracy)/IDS_Datasets/Ready Datasets/CIC_IIoT_2025_preprocessor.joblib
Saved class weights: /home/ibibers/XAI_Evaluation (Exp_Pow_Action_ExpAcc) IEEE TIFS/XAI Evalation For IDS datasets (Explanatory Power,Actionability,Explanation Accuracy)/IDS_Datasets/Ready Datasets/CIC_IIoT_2025_class_weights.json


## 10. Final Verification

In [11]:
print('Final missing values:', int(df_processed.isnull().sum().sum()))
print('Final duplicate rows:', int(df_processed.duplicated().sum()))
print('Final label distribution:')
print(df_processed['label'].value_counts())

Final missing values: 0
Final duplicate rows: 979
Final label distribution:
label
benign_whole-network3                            134183
attack_recon_vuln-scan_whole-network               5869
attack_recon_os-scan_whole-network                 5739
attack_recon_port-scan_whole-network               5227
attack_recon_host-disc-udp-ping_whole-network      5205
                                                  ...  
attack_dos_syn-flood-port-80_yi-camera               32
attack_recon_vuln-scan_plug-vibration                28
attack_dos_slowloris-port-80_yi-camera               25
attack_ddos_slowloris-port-8000_edge1                21
attack_dos_slowloris-port-8000_edge1                 18
Name: count, Length: 937, dtype: int64
